--- Web Scraping Exercise - BeautifulSoup & Pandas
Scraping: https://realpython.github.io/fake-jobs/

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

1. BeautifulSoup object

In [3]:
BASE_URL = "https://realpython.github.io/fake-jobs/"
response = requests.get(BASE_URL)
soup = BeautifulSoup(response.text, "html.parser")

1a. first job title 

In [4]:
first_title_tag = soup.find("h2", class_="title")
print("First job title:", first_title_tag.text.strip())

First job title: Senior Python Developer


1b. Extract all job titles

In [5]:
title_tags = soup.find_all("h2", class_="title")
titles = [tag.text.strip() for tag in title_tags]
print(f"\nFound {len(titles)} job titles. First few:", titles[:3])


Found 100 job titles. First few: ['Senior Python Developer', 'Energy engineer', 'Legal executive']


1c. Extract companies and location

In [6]:
companies = [tag.text.strip() for tag in soup.find_all("h3", class_="company")]
locations = [tag.text.strip() for tag in soup.find_all("p", class_="location")]

In [7]:
dates = [tag.text.strip() for tag in soup.find_all("time")]

1d. Combined DT 

In [8]:
df = pd.DataFrame({
    "title":    titles,
    "company":  companies,
    "location": locations,
    "date":     dates,
})

print("\n── Part 1: DataFrame ──")
print(df.head())



── Part 1: DataFrame ──
                     title                     company              location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

         date  
0  2021-04-08  
1  2021-04-08  
2  2021-04-08  
3  2021-04-08  
4  2021-04-08  


2a. Extract URL

In [9]:
job_cards = soup.find_all("div", class_="card-content")
apply_urls_bs = []
for card in job_cards:
    links = card.find_all("a")
    apply_urls_bs.append(links[-1]["href"])

df["apply_url_bs"] = apply_urls_bs
print("\n── Part 2a: Apply URLs (BeautifulSoup) ──")
print(df["apply_url_bs"].head())


── Part 2a: Apply URLs (BeautifulSoup) ──
0    https://realpython.github.io/fake-jobs/jobs/se...
1    https://realpython.github.io/fake-jobs/jobs/en...
2    https://realpython.github.io/fake-jobs/jobs/le...
3    https://realpython.github.io/fake-jobs/jobs/fi...
4    https://realpython.github.io/fake-jobs/jobs/pr...
Name: apply_url_bs, dtype: object


2b. URLs from extracted data

In [10]:
def make_slug(title):
    slug = title.lower()
    slug = slug.replace(" ", "-")
    slug = re.sub(r"[^a-z0-9-]", "", slug)
    slug = re.sub(r"-+", "-", slug)
    return slug

apply_urls_built = [
    f"https://realpython.github.io/fake-jobs/jobs/{make_slug(title)}-{i}.html"
    for i, title in enumerate(df["title"])
]

df["apply_url_built"] = apply_urls_built

In [11]:
match = df["apply_url_bs"].equals(df["apply_url_built"])
print(f"\n── Part 2b: URLs match BeautifulSoup extraction? {match} ──")
print(df[["title", "apply_url_bs", "apply_url_built"]].head())


── Part 2b: URLs match BeautifulSoup extraction? False ──
                     title                                       apply_url_bs  \
0  Senior Python Developer  https://realpython.github.io/fake-jobs/jobs/se...   
1          Energy engineer  https://realpython.github.io/fake-jobs/jobs/en...   
2          Legal executive  https://realpython.github.io/fake-jobs/jobs/le...   
3   Fitness centre manager  https://realpython.github.io/fake-jobs/jobs/fi...   
4          Product manager  https://realpython.github.io/fake-jobs/jobs/pr...   

                                     apply_url_built  
0  https://realpython.github.io/fake-jobs/jobs/se...  
1  https://realpython.github.io/fake-jobs/jobs/en...  
2  https://realpython.github.io/fake-jobs/jobs/le...  
3  https://realpython.github.io/fake-jobs/jobs/fi...  
4  https://realpython.github.io/fake-jobs/jobs/pr...  


3a. first job page

In [12]:
first_url = "https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html"
resp = requests.get(first_url)
job_soup = BeautifulSoup(resp.text, "html.parser")
description = job_soup.find("p").text.strip()
print("\n── Part 3a: First job description ──")
print(description)


── Part 3a: First job description ──
Fake Jobs for Your Web Scraping Journey


3b. resuable function

In [13]:
def get_job_description(url: str) -> str:
    """
    Fetch a job listing page and return its description paragraph text.

    Args:
        url: Full URL of the job listing page.

    Returns:
        The description text as a clean string.
    """
    resp = requests.get(url)
    page_soup = BeautifulSoup(resp.text, "html.parser")
    return page_soup.find("p").text.strip()

3c. All rows using pandas

In [14]:
url_col = "apply_url_bs" if "apply_url_bs" in df.columns else "apply_url_built"

print(f"\n── Part 3c: Fetching descriptions via '{url_col}' (may take ~30–60 s)… ──")
df["description"] = df[url_col].apply(get_job_description)


── Part 3c: Fetching descriptions via 'apply_url_bs' (may take ~30–60 s)… ──


In [16]:
print("\n── Final DataFrame ──")
print(df[["title", "company", "location", "date", "description"]].head())
print(f"\nShape: {df.shape}")
print("All columns:", df.columns.tolist())



── Final DataFrame ──
                     title                     company              location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

         date                              description  
0  2021-04-08  Fake Jobs for Your Web Scraping Journey  
1  2021-04-08  Fake Jobs for Your Web Scraping Journey  
2  2021-04-08  Fake Jobs for Your Web Scraping Journey  
3  2021-04-08  Fake Jobs for Your Web Scraping Journey  
4  2021-04-08  Fake Jobs for Your Web Scraping Journey  

Shape: (100, 7)
All columns: ['title', 'company', 'location', 'date', 'apply_url_bs', 'apply_url_built', 'description']
